In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# 1. Pipeline Data Engineering Prep
df = pd.read_csv('../data/BrentOilPrices.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
df = df.sort_values('Date').reset_index(drop=True)

# Define time array index and target observation series
time_index = df.index.values
prices = df['Price'].values
n_days = len(prices)

# 2. Build the Bayesian Change Point Model Structure
with pm.Model() as change_point_model:
    
    # Prior for the unknown Switch Point (tau) over all available days
    tau = pm.DiscreteUniform('tau', lower=0, upper=n_days - 1)
    
    # Priors for the expected mean price BEFORE and AFTER the change point
    # We use a broad normal distribution based on global pricing bounds
    mu_1 = pm.Normal('mu_1', mu=prices.mean(), sigma=prices.std())
    mu_2 = pm.Normal('mu_2', mu=prices.mean(), sigma=prices.std())
    
    # Prior for the pricing standard deviation (assumed shared across regimes here)
    sigma = pm.HalfNormal('sigma', sigma=prices.std())
    
    # Use pm.math.switch to assign the correct mean parameter based on the time index
    # If current day index < tau, use mu_1; else use mu_2
    mu_assigned = pm.math.switch(tau > time_index, mu_1, mu_2)
    
    # Define Likelihood function to tie our mathematical priors to observed prices
    likelihood = pm.Normal('y_obs', mu=mu_assigned, sigma=sigma, observed=prices)
    
    # 3. Configure the MCMC Sampler Setup
    print("MCMC Model Structure constructed successfully. Ready for sampling stage.")


In [ ]:
# Note: This block handles output layout and metrics interpretation plots
def plot_model_results(idata, df_data):
    # Summary statistical diagnostic tracking
    summary = az.summary(idata, var_names=["tau", "mu_1", "mu_2", "sigma"])
    print("\n--- Model Convergence Diagnostics ---")
    print(summary)
    
    # Initialize evaluation layout
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # 1. Posterior Trace Distributions
    az.plot_posterior(idata, var_names=['tau'], ax=axes[0], color='purple')
    axes[0].set_title('Posterior Distribution of the Structural Break Date (Tau)')
    
    # 2. Price Parameter Shifts Before vs After
    az.plot_posterior(idata, var_names=['mu_1', 'mu_2'], ax=axes[1])
    axes[1].set_title('Comparison of Regime Pricing Means (USD/Barrel)')
    
    # 3. Raw Data Overlay Mapping
    axes[2].plot(df_data['Date'], df_data['Price'], color='navy', alpha=0.5, label='Observed Price')
    axes[2].set_title('Detected Structural Break Point Overlaid on Historical Brent Prices')
    axes[2].set_ylabel('USD per Barrel')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
